# EDA, limpieza y análisis de datos sobre la vegetación de la Región de Biobío

### Objetivos
* Importar y tabla de la pagina 33 del pdf Catastros_Recursos....
* Explorar el dataset
* Limpiarlo de ser necesario
* Analisis correspondientes

### Fuente
Los datos fueron obtenidos del reporte [Catastro de los Recursos Vegetacionales Nativos de Chile](https://sit.conaf.cl/varios/Catastros_Recursos_Vegetacionales_Nativos_de_Chile_Nov2021.pdf), página 17, cuadro 5.

### Diccionario de variables
| Columna             | Descripción | 
|----------|-------------|
| Región              | Región administrativa del país  |
| Plantación forestal | Superficie cubierta por plataciones forestales en hectáreas (ha)|
| Bosque Nativo       | Superficie cubierta por bosque nativo en hectáreas (ha) |
| Bosque Mixto*       | Superficie cubierta por bosque mixto* en hectáreas (ha) |
| Total               | Superficie cubierta total en hectáreas (ha) |

<small> Bosque nativo corresponde a la mezcla de dos situaciones: 
- Mezcla de bosque nativo (adulto o renoval) y especies plantadas en proporciones que fluctúan entre el 33% al 66% de cubrimiento.
- Bosque nativo con exóticas asilvestradas, es decir, bosque nativo (adulto o renoval) y especies exóticas que se han regenerados en forma natural en proporciones que fluctúan entre 25% y 75% de cubrimiento para cada uno de las categorías que la constituyen.<small>

Verificamos que el texto es extraible usando pdffonts data\Catastros_Recursos_Vegetacionales_Nativos_de_Chile_Nov2021.pdf en una terminal

## 1. Configuración e Inicialización

In [2]:
# Librerias
# Manipulación de datos
import pdfplumber
import pandas as pd

# Utilidades
from pathlib import Path

In [3]:
NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = PROJECT_ROOT / 'data'

### 1.1 Extracción de la tabla desde el pdf

Después definir el pdf del que extraeremos la tabla, usamos pdfplumber para automaticamente encontrar la tabla que usaremos

In [4]:
# esta celda toma tiempo alrededor de 30seg
DATA_FILE = DATA_DIR / 'Catastros_Recursos_Vegetacionales_Nativos_de_Chile_Nov2021.pdf'

with pdfplumber.open(DATA_FILE) as pdf:
    texto_completo = ""
    for i, page in enumerate(pdf.pages):
        texto = page.extract_text() or ''
        if 'Cuadro 5' in texto:  # o el título exacto de tu tabla
            print(f'Encontrado en página índice {i} (página real {i+1})')

Encontrado en página índice 14 (página real 15)
Encontrado en página índice 16 (página real 17)


Aqui imprimimos la tabla que identificamos en la celda anterior

In [5]:
with pdfplumber.open(DATA_FILE) as pdf:
    page = pdf.pages[16]  # índice 0-based, página real 17
    tabla = page.extract_tables()
    
    for fila in tabla[0]:
        print(fila)

['Tarapacá', '29.264,2', '33.246,4', '0,0', '62.510,6']
[None, '3.049,9', '11.899,3', '0,0', '14.949,2']
['Atacama', '276,4', '3.224,2', '0,0', '3.500,6']
[None, '12.285,1', '48,474,9', '886,4', '61.646,4']
['Valparaíso', '68.757,9', '484.115,7', '724,6', '553.598,1']
[None, '9.181,0', '363.955,3', '218,0', '373.354,3']
['O’Higgins', '130.536,4', '459.309,1', '545,7', '590.391,2']
[None, '634.893,5', '581.515,3', '28.674,8', '1.245.083,6']
['Ñuble', '380.714,9', '247.979,8', '17.051,8', '645.746,5']
[None, '875.178,4', '597.572,7', '51.635,9', '1.524.387,0']
['La Araucanía', '632.289,0', '964.152,9', '47.639,5', '1.644.081,3']
[None, '208.775,2', '908.530,7', '17.866,0', '1.135.171,9']
['Los Lagos', '96.598,8', '2.827.436,4', '12.799,3', '2.936.834,5']
[None, '32.017,3', '4.398.744,8', '1.083,0', '4.431.845,1']
['Magallanes y de\nla Antártica', '286,2', '2.760.176,6', '0,0', '2.760.462,8']
[None, '', '', '', '']
[None, '', '', '', '']
['Total', '3.114.125', '14.737.485', '179.125', '18

La gran mayoria de los datos númericos fueron extraidos, sin embargo la primera linea "Arica y Parinacota" y algunas regiones no aparecen en la tabla. Dado que el dataset es pequeño, escribiremos a mano la información usando de base lo extraido en la celda anterior. 

In [6]:
LIMPIO_FILE = DATA_DIR / 'bosque_chile_excel.csv'


datos = [
    ['Arica y Parinacota', '20,8', '470151,3', '0,0', '47.172,1'],
    ['Tarapacá', '29.264,2', '33.246,4', '0,0', '62.510,6'],
    ['Antofagasta', '3.049,9', '11.899,3', '0,0', '14.949,2'],
    ['Atacama', '276,4', '3.224,2', '0,0', '3.500,6'],
    ['Coquimbo', '12.285,1', '48.474,9', '886,4', '61.646,4'],
    ['Valparaíso', '68.757,9', '484.115,7', '724,6', '553.598,1'],
    ['Metropolitana', '9.181,0', '363.955,3', '218,0', '373.354,3'],
    ["O'Higgins", '130.536,4', '459.309,1', '545,7', '590.391,2'],
    ['Maule', '634.893,5', '581.515,3', '28.674,8', '1.245.083,6'],
    ['Ñuble', '380.714,9', '247.979,8', '17.051,8', '645.746,5'],
    ['Biobío', '875.178,4', '597.572,7', '51.635,9', '1.524.387,0'],
    ['La Araucanía', '632.289,0', '964.152,9', '47.639,5', '1.644.081,3'],
    ['Los Ríos', '208.775,2', '908.530,7', '17.866,0', '1.135.171,9'],
    ['Los Lagos', '96.598,8', '2.827.436,4', '12.799,3', '2.936.834,5'],
    ['Aysén', '32.017,3', '4.398.744,8', '1.083,0', '4.431.845,1'],
    ['Magallanes y de la Antártica', '286,2', '2.760.176,6', '0,0', '2.760.462,8'],
    ['Total', '3.114.125', '14.737.485', '179.125', '18.030.735'],
    ['Porcentaje Nacional', '17,27', '81,74', '0,99', '100']
]

raw_df = pd.DataFrame(datos, columns=["region", "plantacion_forestal", "bosque_nativo", "bosque_mixto", "total"])
raw_df.head()

,region,plantacion_forestal,bosque_nativo,bosque_mixto,total
0,Arica y Parinacota,"20,8","470151,3","0,0","47.172,1"
1,Tarapacá,"29.264,2","33.246,4","0,0","62.510,6"
2,Antofagasta,"3.049,9","11.899,3","0,0","14.949,2"
3,Atacama,"276,4","3.224,2","0,0","3.500,6"
4,Coquimbo,"12.285,1","48.474,9","886,4","61.646,4"


## 2. Análisis exploratorio de datos (EDA)

### 2.2 Cargar el dataset a usar y vista preliminar

In [7]:
df = raw_df.copy()

df.head()

,region,plantacion_forestal,bosque_nativo,bosque_mixto,total
0,Arica y Parinacota,"20,8","470151,3","0,0","47.172,1"
1,Tarapacá,"29.264,2","33.246,4","0,0","62.510,6"
2,Antofagasta,"3.049,9","11.899,3","0,0","14.949,2"
3,Atacama,"276,4","3.224,2","0,0","3.500,6"
4,Coquimbo,"12.285,1","48.474,9","886,4","61.646,4"


### 2.3 Estructura de los datos

In [8]:
print(df.shape)
df.info

(18, 5)


<bound method DataFrame.info of                           region plantacion_forestal bosque_nativo  \
0             Arica y Parinacota                20,8      470151,3   
1                       Tarapacá            29.264,2      33.246,4   
2                    Antofagasta             3.049,9      11.899,3   
3                        Atacama               276,4       3.224,2   
4                       Coquimbo            12.285,1      48.474,9   
5                     Valparaíso            68.757,9     484.115,7   
6                  Metropolitana             9.181,0     363.955,3   
7                      O'Higgins           130.536,4     459.309,1   
8                          Maule           634.893,5     581.515,3   
9                          Ñuble           380.714,9     247.979,8   
10                        Biobío           875.178,4     597.572,7   
11                  La Araucanía           632.289,0     964.152,9   
12                      Los Ríos           208.775,2     9

#### Inspección inicial

Al revisar la estructura del dataset, observamos lo siguiente:

- La tabla contiene todas las regiones del país (16) más dos filas que incluyen el total de cada tipo de bosque y su porcentaje acumulativo.
- Los valores de las columnas `plantacion_forestal`,`bosque_nativo`,`bosque_mixto` y `total` son valores string ya que tienen puntos.

Esto último lo verificaremos en la siguiente sección

### 2.4 Tipos de datos

In [9]:
df.dtypes.to_frame().T

,region,plantacion_forestal,bosque_nativo,bosque_mixto,total
0,str,str,str,str,str


Efectivamente, los valores que son númericos estan guardados como texto.

In [10]:
columnas = ["region", "plantacion_forestal", "bosque_nativo", "bosque_mixto", "total"]
for col in columnas:
    print(f"\n{col}:")
    print(df[col].head(10).tolist())


region:
['Arica y Parinacota', 'Tarapacá', 'Antofagasta', 'Atacama', 'Coquimbo', 'Valparaíso', 'Metropolitana', "O'Higgins", 'Maule', 'Ñuble']

plantacion_forestal:
['20,8', '29.264,2', '3.049,9', '276,4', '12.285,1', '68.757,9', '9.181,0', '130.536,4', '634.893,5', '380.714,9']

bosque_nativo:
['470151,3', '33.246,4', '11.899,3', '3.224,2', '48.474,9', '484.115,7', '363.955,3', '459.309,1', '581.515,3', '247.979,8']

bosque_mixto:
['0,0', '0,0', '0,0', '0,0', '886,4', '724,6', '218,0', '545,7', '28.674,8', '17.051,8']

total:
['47.172,1', '62.510,6', '14.949,2', '3.500,6', '61.646,4', '553.598,1', '373.354,3', '590.391,2', '1.245.083,6', '645.746,5']


Dado que los nombres de las regiones son faciles de leer como estan, procederemos a limpiar los datos manteniendo los valores de la columna `region`.

## 3. Limpieza

### 3.1 Cambiar valores texto a número si corresponde

In [11]:
def limpiar_numero_cl(serie):
    return serie.str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)

columnas_num = ['plantacion_forestal', 'bosque_nativo', 'bosque_mixto', 'total']

for col in columnas_num:
    if df[col].dtype == 'object' or df[col].dtype == 'str':  # solo si todavía es texto
        df[col] = limpiar_numero_cl(df[col])

print(df.dtypes.to_frame().T)
df.head()

  region plantacion_forestal bosque_nativo bosque_mixto    total
0    str             float64       float64      float64  float64


,region,plantacion_forestal,bosque_nativo,bosque_mixto,total
0,Arica y Parinacota,20.8,470151.3,0.0,47172.1
1,Tarapacá,29264.2,33246.4,0.0,62510.6
2,Antofagasta,3049.9,11899.3,0.0,14949.2
3,Atacama,276.4,3224.2,0.0,3500.6
4,Coquimbo,12285.1,48474.9,886.4,61646.4


### 3.2 Verificar ausencia de nulos y duplicados

In [12]:
# Verificar nulos y duplicados

nulos = df.isna().sum()
duplicados = df['region'].duplicated().sum()

print('Comunas duplicadas ' + str(duplicados))
print('Numero de nulos por columna')
nulos.to_frame().T

Comunas duplicadas 0
Numero de nulos por columna


,region,plantacion_forestal,bosque_nativo,bosque_mixto,total
0,0,0,0,0,0


#### Verificación

Tras la limpieza
- Las columnas `plantacion_forestal`,`bosque_nativo`,`bosque_mixto` y `total` son columnas númericas.
- No existen nulos ni duplicados en el dataset.

## Continuación análisis exploratorio

### 2.4 Descripción de datos

In [13]:
# Este es el df sin las filas total y porcentaje nacional al igual que sin la columna total
df_vegetacion = df[~df['region'].isin(['Total', 'Porcentaje Nacional'])].reset_index(drop=True)

df_vegetacion[-5:]

,region,plantacion_forestal,bosque_nativo,bosque_mixto,total
11,La Araucanía,632289.0,964152.9,47639.5,1644081.3
12,Los Ríos,208775.2,908530.7,17866.0,1135171.9
13,Los Lagos,96598.8,2827436.4,12799.3,2936834.5
14,Aysén,32017.3,4398744.8,1083.0,4431845.1
15,Magallanes y de la Antártica,286.2,2760176.6,0.0,2760462.8


In [14]:
resumen = df_vegetacion.describe()

resumen[columnas_num] = resumen[columnas_num].map(lambda x: f'{x:,.2f}')

resumen

,plantacion_forestal,bosque_nativo,bosque_mixto,total
count,16.00,16.00,16.00,16.00
mean,"194,632.81","947,530.34","11,195.31","1,126,920.95"
std,"281,005.49","1,262,566.67","17,376.77","1,282,039.53"
min,20.80,"3,224.20",0.00,"3,500.60"
25%,"7,648.23","198,103.57",0.00,"62,294.55"
50%,"50,387.60","477,133.50",805.50,"618,068.85"
75%,"251,760.12","922,436.25","17,255.35","1,554,310.57"
max,"875,178.40","4,398,744.80","51,635.90","4,431,845.10"


In [15]:
top5 = df_vegetacion.nlargest(5, 'total')[columnas].reset_index(drop=True)
bottom5 = df_vegetacion.nsmallest(5, 'total')[columnas].reset_index(drop=True)

comparacion = pd.concat(
    [top5.rename(columns={
            "region": "Top 5 regiones"}),bottom5.rename(columns={
            "region": "Bottom 5 regiones"})],axis=1)

comparacion

,Top 5 regiones,plantacion_forestal,bosque_nativo,bosque_mixto,total,Bottom 5 regiones,plantacion_forestal,bosque_nativo,bosque_mixto,total
0,Aysén,32017.3,4398744.8,1083.0,4431845.1,Atacama,276.4,3224.2,0.0,3500.6
1,Los Lagos,96598.8,2827436.4,12799.3,2936834.5,Antofagasta,3049.9,11899.3,0.0,14949.2
2,Magallanes y de la Antártica,286.2,2760176.6,0.0,2760462.8,Arica y Parinacota,20.8,470151.3,0.0,47172.1
3,La Araucanía,632289.0,964152.9,47639.5,1644081.3,Coquimbo,12285.1,48474.9,886.4,61646.4
4,Biobío,875178.4,597572.7,51635.9,1524387.0,Tarapacá,29264.2,33246.4,0.0,62510.6


#### Conclusiones

A partir de las estadísticas descriptivas, podemos observar:

- Existe una gran variación en todas las variables. Esto se debe a que chile es un país geograficamente distinto, sectores deserticos como Arica y Parinacota, Antofagasta y Coquimbo tienen menos superficie forestal que las regiones del sur de Chile (Top 5).
- La región estudiada, Región del Biobío, es la 4 cuarta con más superficie plantada del país. Esto la hace sumamente relevante para estudiar ya que la vegetación es un combustible para incendios y entre más disponible este, mayor es el riego que pueda alcanzar características ignibles.

### Top 5 regiones con más superficie de cata tipo de plantación

In [16]:
top5_nativo = df_vegetacion.nlargest(5, 'bosque_nativo')[['region','bosque_nativo']].reset_index(drop=True)
top5_forestal = df_vegetacion.nlargest(5, 'plantacion_forestal')[['region','plantacion_forestal']].reset_index(drop = True)
top5_mixto = df_vegetacion.nlargest(5, 'bosque_mixto')[['region','bosque_mixto']].reset_index(drop=True)

comparacion = pd.concat(
    [top5_nativo.rename(columns={
            "region": "Top 5 regiones Nativo"}),
     top5_forestal.rename(columns={
            "region": "Top 5 regiones Forestal"}), 
     top5_mixto.rename(columns={
            "region": "Top 5 regiones Mixto"})],axis=1)

comparacion

,Top 5 regiones Nativo,bosque_nativo,Top 5 regiones Forestal,plantacion_forestal,Top 5 regiones Mixto,bosque_mixto
0,Aysén,4398744.8,Biobío,875178.4,Biobío,51635.9
1,Los Lagos,2827436.4,Maule,634893.5,La Araucanía,47639.5
2,Magallanes y de la Antártica,2760176.6,La Araucanía,632289.0,Maule,28674.8
3,La Araucanía,964152.9,Ñuble,380714.9,Los Ríos,17866.0
4,Los Ríos,908530.7,Los Ríos,208775.2,Ñuble,17051.8


Observaciones:

Como podemos ver, la Región del Biobío presenta la mayor cantidad de superficie dedicada a plantaciones forestales y a bosque mixto del país. Esto reafirma la hipotesis que la región es especialmente vulnerable a incendios.

### Guardar dataset limpio

In [17]:
# Guardando el csv limpio con las columnas que sabemos que queremos

LIMPIO_FILE = DATA_DIR / 'bosques_chile_excel.csv'

df.to_csv(LIMPIO_FILE, index=False)